# 14 — Final Results Tables and Figures

This notebook regenerates the core results used in the final research report and saves consolidated tables/figures under `results/`. It is intentionally reporting-focused: no new specification is introduced here.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel
from src.inference import primary_and_robustness_hac
from src.concentration import calculate_monthly_concentration
from src.conditioning import merge_performance_and_concentration, primary_top10_spec, hhi_spec, lag_concentration_one_month
from src.mega_cap import exclude_top_permco, compare_spreads, delta_hac_summary
from src.mega_cap_decomposition import fixed_rank_value_weighted_returns, fixed_rank_spread, decomposition_table, mega_cap_leg_diagnostics, component_hac_summary
from src.industry_neutral import attach_ff49, industry_demeaned_momentum, assign_quintiles_from_signal
from src.daily_data import read_daily_stock, read_daily_market, read_fama_french_daily_rf, align_market_and_rf
from src.beta import BetaConfig, estimate_monthly_capm_beta
from src.beta_neutral import build_beta_neutral_momentum, ex_post_market_beta

TABLE_DIR = ROOT / 'results/tables'
FIG_DIR = ROOT / 'results/figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Rebuild the common monthly research objects


In [ ]:
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
DAILY_STOCK = ROOT / 'data/raw/daily_stock_naq_1998_2025_prod_v1.csv.gz'
DAILY_MARKET = ROOT / 'data/raw/daily_crsp_vw_market_1998_2025_v1.csv.gz'
FF_DAILY = ROOT / 'data/raw/F-F_Research_Data_Factors_daily.csv'

clean, clean_audit = build_clean_monthly_panel(MONTHLY, CleaningConfig())
history = build_monthly_history_panel(MONTHLY)
market_state = build_market_state_panel(MONTHLY)
conc = calculate_monthly_concentration(market_state)

mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)
rank_ic = monthly_rank_ic(mom)
qret = quintile_returns(mom)
vw = quintile_return_panel(qret, return_col='VW_Return')
ew = quintile_return_panel(qret, return_col='EW_Return')

print('Clean formation rows:', len(clean))
print('Unique PERMNO:', clean['PERMNO'].nunique())
print('Valid momentum:', mom['Momentum_12_2'].notna().sum())
print('Months:', clean['MthCalDt'].nunique())


## 2. Table 1 — Sample and baseline momentum


In [ ]:
vw_hac = primary_and_robustness_hac(vw['Q5_minus_Q1'])
ew_hac = primary_and_robustness_hac(ew['Q5_minus_Q1'])
ic_hac = primary_and_robustness_hac(rank_ic['RankIC'])

monthly_n = clean.groupby('MthCalDt')['PERMNO'].nunique()
table1 = pd.DataFrame({
    'Metric': [
        'Sample start','Sample end','Formation rows','Unique PERMNO','Formation months',
        'Valid momentum observations','Average monthly investable stocks','Median monthly investable stocks',
        'Mean Rank IC','Mean VW Q5-Q1','Mean EW Q5-Q1'
    ],
    'Value': [
        clean['MthCalDt'].min().date(), clean['MthCalDt'].max().date(), len(clean), clean['PERMNO'].nunique(), clean['MthCalDt'].nunique(),
        mom['Momentum_12_2'].notna().sum(), monthly_n.mean(), monthly_n.median(),
        rank_ic['RankIC'].mean(), vw['Q5_minus_Q1'].mean(), ew['Q5_minus_Q1'].mean()
    ]
})
table1.to_csv(TABLE_DIR / 'table1_sample_baseline.csv', index=False)
baseline_inference = pd.concat({
    'RankIC': ic_hac,
    'VW_Q5_minus_Q1': vw_hac,
    'EW_Q5_minus_Q1': ew_hac,
}, names=['Outcome','Inference'])
baseline_inference.to_csv(TABLE_DIR / 'table1_baseline_inference.csv')
display(table1)
display(baseline_inference)


## 3. Table 2 — Aggregate concentration tests


In [ ]:
primary = merge_performance_and_concentration(vw, conc)
ew_conc = merge_performance_and_concentration(ew, conc)
ic_conc = merge_performance_and_concentration(rank_ic[['MthCalDt','RankIC']], conc)

r1,_ = primary_top10_spec(primary, maxlags=6)
r2,_ = primary_top10_spec(primary, maxlags=6, add_time_trend=True)
r3,_ = hhi_spec(primary, maxlags=6)
r4,_ = primary_top10_spec(ew_conc, maxlags=6)
r5,_ = primary_top10_spec(ic_conc, outcome_col='RankIC', maxlags=6)

lagged = lag_concentration_one_month(conc)
lagged = lagged.rename(columns={'Top10Share_Lag1':'Top10Share'}) if 'Top10Share_Lag1' in lagged.columns else lagged
lag_cols = [c for c in lagged.columns if 'Top10Share' in c and c != 'Top10Share']
if lag_cols:
    lagged = lagged.rename(columns={lag_cols[0]:'Top10Share'})
lag_merge = vw[['MthCalDt','Q5_minus_Q1']].merge(lagged[['MthCalDt','Top10Share']], on='MthCalDt', how='inner')
r6,_ = primary_top10_spec(lag_merge, maxlags=6)

def row_from_reg(label, reg, term):
    s = reg.loc[term]
    return {'Specification':label,'Estimate':s['Estimate'],'HAC_SE':s['HAC_SE'],'T_Stat':s['T_Stat'],'P_Value':s['P_Value'],'CI95_Low':s['CI95_Low'],'CI95_High':s['CI95_High'],'N':s['N']}

table2 = pd.DataFrame([
    row_from_reg('Top10 -> VW spread', r1, 'Top10Share_10pp'),
    row_from_reg('Top10 -> VW spread + linear trend', r2, 'Top10Share_10pp'),
    row_from_reg('HHI -> VW spread', r3, 'HHI_0p01'),
    row_from_reg('Top10 -> EW spread', r4, 'Top10Share_10pp'),
    row_from_reg('Top10 -> Rank IC', r5, 'Top10Share_10pp'),
    row_from_reg('Lagged Top10 -> VW spread', r6, 'Top10Share_10pp'),
])
table2.to_csv(TABLE_DIR / 'table2_concentration_tests.csv', index=False)
display(table2)


## 4. Table 3 — Mega-cap exclusion and decomposition


In [ ]:
mega_rows = []
vw_alt_store = {}
for n in [5,10,20]:
    c_alt = exclude_top_permco(clean, market_state, n=n)
    m_alt = assign_signal_quintiles(attach_momentum_to_formation_universe(c_alt, history))
    v_alt = quintile_return_panel(quintile_returns(m_alt), return_col='VW_Return')
    vw_alt_store[n] = v_alt
    cmp = compare_spreads(vw, v_alt, alt_label=f'ExTop{n}')
    hac = delta_hac_summary(cmp, alt_label=f'ExTop{n}')
    mega_rows.append({
        'Specification':f'ExTop{n}',
        'MeanSpread':v_alt['Q5_minus_Q1'].mean(),
        'MeanDelta':cmp[f'Delta_ExTop{n}_Minus_Baseline'].mean(),
        'HAC6_t':hac.loc['Primary_HAC6','T_Stat'],
        'HAC6_p':hac.loc['Primary_HAC6','P_Value'],
        'HAC12_p':hac.loc['Robustness_HAC12','P_Value'],
    })

table3a = pd.DataFrame(mega_rows)

fixed_qret = fixed_rank_value_weighted_returns(mom, market_state, n=10)
vw_fixed = fixed_rank_spread(fixed_qret)
decomp = decomposition_table(vw, vw_fixed, vw_alt_store[10])
decomp_hac = component_hac_summary(decomp)
table3b = pd.DataFrame([
    {'Component':c,'Mean':decomp[c].mean(),'HAC6_t':decomp_hac[c].loc['Primary_HAC6','T_Stat'],'HAC6_p':decomp_hac[c].loc['Primary_HAC6','P_Value'],'HAC12_p':decomp_hac[c].loc['Robustness_HAC12','P_Value']}
    for c in ['DirectWeightEffect','ReRankingEffect','TotalChange']
])

diag = mega_cap_leg_diagnostics(mom, market_state, n=10)
legs = diag[diag['SignalQuintile'].isin([1,5])].copy()
table3c = legs.groupby('SignalQuintile', observed=True)[['FormationWeightShare','ObservedWeightShare','ReturnContribution','TopN_SecurityCount']].agg(['mean','median'])

table3a.to_csv(TABLE_DIR / 'table3a_mega_cap_exclusion.csv', index=False)
table3b.to_csv(TABLE_DIR / 'table3b_mega_cap_decomposition.csv', index=False)
table3c.to_csv(TABLE_DIR / 'table3c_mega_cap_leg_diagnostics.csv')
display(table3a)
display(table3b)
display(table3c)


## 5. Table 4A — Industry neutralisation


In [ ]:
mom_ff = attach_ff49(mom)
mom_neutral = industry_demeaned_momentum(mom_ff, min_industry_size=10)
mom_neutral = assign_quintiles_from_signal(mom_neutral, 'Momentum_IndustryNeutral')
vw_neutral = quintile_return_panel(quintile_returns(mom_neutral), return_col='VW_Return')
ic_neutral = monthly_rank_ic(mom_neutral, signal_col='Momentum_IndustryNeutral')

ic_cmp = rank_ic[['MthCalDt','RankIC']].rename(columns={'RankIC':'Raw'}).merge(ic_neutral[['MthCalDt','RankIC']].rename(columns={'RankIC':'Neutral'}), on='MthCalDt')
ic_cmp['Delta'] = ic_cmp['Neutral'] - ic_cmp['Raw']
vw_cmp = vw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Raw'}).merge(vw_neutral[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Neutral'}), on='MthCalDt')
vw_cmp['Delta'] = vw_cmp['Neutral'] - vw_cmp['Raw']

vw_ind_hac = primary_and_robustness_hac(vw_cmp['Delta'])
ic_ind_hac = primary_and_robustness_hac(ic_cmp['Delta'])
table4a = pd.DataFrame([
    {'Outcome':'VW spread','RawMean':vw_cmp['Raw'].mean(),'NeutralMean':vw_cmp['Neutral'].mean(),'Delta':vw_cmp['Delta'].mean(),'HAC6_t':vw_ind_hac.loc['Primary_HAC6','T_Stat'],'HAC6_p':vw_ind_hac.loc['Primary_HAC6','P_Value'],'HAC12_p':vw_ind_hac.loc['Robustness_HAC12','P_Value']},
    {'Outcome':'Rank IC','RawMean':ic_cmp['Raw'].mean(),'NeutralMean':ic_cmp['Neutral'].mean(),'Delta':ic_cmp['Delta'].mean(),'HAC6_t':ic_ind_hac.loc['Primary_HAC6','T_Stat'],'HAC6_p':ic_ind_hac.loc['Primary_HAC6','P_Value'],'HAC12_p':ic_ind_hac.loc['Robustness_HAC12','P_Value']},
])
table4a.to_csv(TABLE_DIR / 'table4a_industry_neutral.csv', index=False)
display(table4a)


## 6. Table 4B — Beta neutralisation


In [ ]:
daily_stock = read_daily_stock(DAILY_STOCK)
market = read_daily_market(DAILY_MARKET)
rf = read_fama_french_daily_rf(FF_DAILY)
market_rf, _ = align_market_and_rf(market, rf)
beta, beta_audit = estimate_monthly_capm_beta(clean[['PERMNO','MthCalDt']], daily_stock, market_rf, BetaConfig(252,5,126))
beta_panel = build_beta_neutral_momentum(mom, beta, market_rf)

bn_hac = primary_and_robustness_hac(beta_panel['NeutralisationEffect'])
scale_hac = primary_and_robustness_hac(beta_panel['ScaledNeutralisationEffect'])
expost_raw = ex_post_market_beta(beta_panel, 'OriginalSpread', hac_lag=6)
expost_bn = ex_post_market_beta(beta_panel, 'BetaNeutralSpread', hac_lag=6)

table4b = pd.DataFrame([
    {'Metric':'Original VW spread','Mean':beta_panel['OriginalSpread'].mean()},
    {'Metric':'Beta-eligible VW spread','Mean':beta_panel['EligibleSpread'].mean()},
    {'Metric':'Beta-neutral VW spread','Mean':beta_panel['BetaNeutralSpread'].mean()},
    {'Metric':'Neutralisation effect','Mean':beta_panel['NeutralisationEffect'].mean(),'HAC6_t':bn_hac.loc['Primary_HAC6','T_Stat'],'HAC6_p':bn_hac.loc['Primary_HAC6','P_Value'],'HAC12_p':bn_hac.loc['Robustness_HAC12','P_Value']},
    {'Metric':'Leg-rescaled neutralisation effect','Mean':beta_panel['ScaledNeutralisationEffect'].mean(),'HAC6_t':scale_hac.loc['Primary_HAC6','T_Stat'],'HAC6_p':scale_hac.loc['Primary_HAC6','P_Value'],'HAC12_p':scale_hac.loc['Robustness_HAC12','P_Value']},
    {'Metric':'Ex-post market beta: raw','Mean':expost_raw['MarketBeta'],'HAC6_t':expost_raw['Beta_T'],'HAC6_p':expost_raw['Beta_P']},
    {'Metric':'Ex-post market beta: beta-neutral','Mean':expost_bn['MarketBeta'],'HAC6_t':expost_bn['Beta_T'],'HAC6_p':expost_bn['Beta_P']},
])
table4b.to_csv(TABLE_DIR / 'table4b_beta_neutral.csv', index=False)
display(table4b)


## 7. Figure 1 — Market concentration


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(conc['MthCalDt'], 100*conc['Top10Share'])
ax.set_title('Top-10 Company Share of US Equity Market Capitalisation')
ax.set_xlabel('Formation month')
ax.set_ylabel('Top-10 share (%)')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure1_top10_concentration.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Figure 2 — Cumulative baseline momentum spreads


In [ ]:
cum = vw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'VW'}).merge(ew[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'EW'}), on='MthCalDt')
cum['VW_Cumulative'] = (1+cum['VW'].fillna(0)).cumprod()-1
cum['EW_Cumulative'] = (1+cum['EW'].fillna(0)).cumprod()-1
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(cum['MthCalDt'], cum['VW_Cumulative'], label='Value-weighted')
ax.plot(cum['MthCalDt'], cum['EW_Cumulative'], label='Equal-weighted')
ax.set_title('Cumulative Q5-Q1 Momentum Spread')
ax.set_xlabel('Formation month')
ax.set_ylabel('Cumulative return')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure2_cumulative_momentum.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Figure 3 — Baseline vs Ex-Top10 cumulative spread


In [ ]:
f3 = vw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Baseline'}).merge(vw_alt_store[10][['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'ExTop10'}), on='MthCalDt')
f3['BaselineCum']=(1+f3['Baseline'].fillna(0)).cumprod()-1
f3['ExTop10Cum']=(1+f3['ExTop10'].fillna(0)).cumprod()-1
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(f3['MthCalDt'], f3['BaselineCum'], label='Baseline VW')
ax.plot(f3['MthCalDt'], f3['ExTop10Cum'], label='Ex-Top10 VW')
ax.set_title('Momentum Implementation With and Without Top-10 Companies')
ax.set_xlabel('Formation month')
ax.set_ylabel('Cumulative Q5-Q1 return')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure3_ex_top10_cumulative.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. Figure 4 — Mega-cap decomposition


In [ ]:
plot_decomp = table3b[['Component','Mean']].copy()
fig, ax = plt.subplots(figsize=(7,5))
ax.bar(plot_decomp['Component'], 100*plot_decomp['Mean'])
ax.axhline(0, linewidth=1)
ax.set_title('Top-10 Exclusion: Mean Monthly Decomposition')
ax.set_ylabel('Effect on Q5-Q1 return (percentage points/month)')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure4_mega_cap_decomposition.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. Figure 5 — Ex-ante momentum beta and realised beta summary


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(beta_panel['MthCalDt'], beta_panel['Beta_LS'])
ax.axhline(0, linewidth=1)
ax.set_title('Ex-Ante Net Market Beta of the VW Momentum Spread')
ax.set_xlabel('Formation month')
ax.set_ylabel('Estimated beta')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure5_ex_ante_beta.png', dpi=200, bbox_inches='tight')
plt.show()

pd.DataFrame({
    'Portfolio':['Raw momentum','Beta-neutral momentum'],
    'ExPostMarketBeta':[expost_raw['MarketBeta'], expost_bn['MarketBeta']],
    'HAC6_p':[expost_raw['Beta_P'], expost_bn['Beta_P']],
})


## 12. Reporting checklist

Before freezing the paper, verify that all CSV tables and PNG figures were written to `results/`. The final prose should use these consolidated outputs rather than manually transcribed notebook screenshots.


In [ ]:
print('Tables:')
for p in sorted(TABLE_DIR.glob('*.csv')):
    print(' -', p.name)
print('\nFigures:')
for p in sorted(FIG_DIR.glob('*.png')):
    print(' -', p.name)
